Imports

In [ ]:
# =========================
# REPRODUCIBILITY + IMPORTS
# =========================
from pathlib import Path
import os, random
import json, csv
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from PIL import Image

# -------------------------
# Global seed / determinism
# -------------------------
SEED = 42

def seed_everything(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

seed_everything(SEED, deterministic=True)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =========================
# CONFIG
# =========================
PROCESSED_ROOT = Path("../../data/processed/Stage1/gray")

# IMPORTANT: use your train1/train2/train3 rewritten index
INDEX_CSV = PROCESSED_ROOT / "index_train123.csv"

TRAIN_LABELS_JSON = Path("../../data/labels/Stage1/gray/combined/train.json")
VAL_LABELS_JSON   = Path("../../data/labels/Stage1/gray/combined/val.json")

NUM_CLASSES = 2
BATCH_SIZE  = 40
LR_HEAD     = 2e-3
LR_FULL     = 2e-4
EPOCHS      = 25
IMAGE_SIZE  = 256
WEIGHT_DECAY = 1e-4

UNFREEZE_EPOCH = 3   # try 3–6
IMAGE_MODE = "gray"   # "gray" or "rgb"

MODEL_OUT_PATH = Path("../../models/classifier/Stage1/model.pt")
MODEL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# LOAD INDEX CSV
# =========================
def read_index_csv(index_csv_path: Path):
    rows = []
    with open(index_csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    if not rows:
        raise ValueError(f"index.csv is empty: {index_csv_path}")
    if "filepath" not in rows[0]:
        raise ValueError("index.csv must contain a 'filepath' column")
    return rows

index_rows = read_index_csv(INDEX_CSV)
print("Rows in index.csv:", len(index_rows))
print("Example row keys:", list(index_rows[0].keys()))
print("Example filepath:", index_rows[0]["filepath"])

# =========================
# LOAD LABEL MAPS
# =========================
def load_label_map(label_path: Path):
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    with open(label_path, "r") as f:
        data = json.load(f)

    out = {}
    bad = []

    if isinstance(data, dict):
        for k, v in data.items():
            k_norm = k.replace("\\", "/")
            out[k_norm] = int(v)
            out[Path(k_norm).name] = int(v)
        return out

    if isinstance(data, list):
        for item in data:
            filepath = item.get("image", "").replace("\\", "/")
            if not filepath:
                bad.append(("MISSING_IMAGE_FIELD", item))
                continue

            fname = Path(filepath).name
            empty = item.get("empty", None)
            if empty is None:
                bad.append((filepath, "empty=None"))
                continue

            empty = int(empty)
            if empty == 1:
                class_id = 0
            elif empty == 0:
                class_id = 1
            else:
                bad.append((filepath, f"empty={empty}"))
                continue

            out[filepath] = class_id
            out[fname] = class_id

        if bad:
            print("Found inconsistent/bad label rows (showing up to 10):")
            for row in bad[:10]:
                print("  ", row)
            raise ValueError(f"Inconsistent labels for {len(bad)} samples in {label_path}")

        return out

    raise ValueError("Label JSON must be a dict or list")

train_label_map = load_label_map(TRAIN_LABELS_JSON)
val_label_map   = load_label_map(VAL_LABELS_JSON)

print("Train label distribution:", Counter(train_label_map.values()))
print("Val label distribution  :", Counter(val_label_map.values()))

# =========================
# CROPS / AUGS
# =========================
def crop_to_tray_interior(img: Image.Image) -> Image.Image:
    w, h = img.size
    return img.crop((
        int(w * 0.35),
        int(h * 0.35),
        int(w * 0.65),
        int(h * 0.65),
    ))

class RandomBorderZero:
    def __init__(self, p=0.7, min_frac=0.03, max_frac=0.10):
        self.p = p
        self.min_frac = min_frac
        self.max_frac = max_frac

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if torch.rand(1).item() > self.p:
            return x
        _, h, w = x.shape
        frac = float(torch.empty(1).uniform_(self.min_frac, self.max_frac))
        bx = int(w * frac)
        by = int(h * frac)
        x = x.clone()
        x[:, :by, :] = 0
        x[:, h-by:, :] = 0
        x[:, :, :bx] = 0
        x[:, :, w-bx:] = 0
        return x

# =========================
# DATASET 
# =========================
_MISS = object()

class ProcessedSplitDataset(Dataset):
    def __init__(
        self,
        index_rows,
        processed_root: Path,
        split: str,
        label_map: dict,
        transform=None,
        auto_label_stage1_missing: bool = False,  #  safer default (no silent bias)
        stage1_default_class: int = 1,
        verify_files_exist: bool = True,          #  catch bad paths early
    ):
        self.processed_root = processed_root
        self.split = split.lower().strip()
        self.transform = transform
        self.label_map = label_map

        # ---- select rows for this split ----
        rows = []
        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            if not fp:
                continue

            row_split = (r.get("split") or "").strip().lower()
            if row_split:
                if row_split == self.split:
                    rows.append(r)
            else:
                if f"images/{self.split}/" in fp:
                    rows.append(r)

        if not rows:
            raise ValueError(f"No samples found for split='{self.split}'")

        self.filepaths = []
        self.labels = []

        missing_hard = []
        missing_files = []
        auto_filled = 0

        for r in rows:
            fp_norm = (r.get("filepath") or "").replace("\\", "/").strip()
            fname = Path(fp_norm).name
            stage = (r.get("stage") or "").strip().lower()

            # train1/train2/train3 should map to "train" labels filepaths
            label_split = "train" if self.split in {"train1", "train2", "train3"} else self.split

            # IMPORTANT: do NOT use `or` because label 0 is falsy.
            keys_to_try = [
                fp_norm,
                f"images/{label_split}/{fname}",
                fname,
            ]

            label = _MISS
            for k in keys_to_try:
                label = self.label_map.get(k, _MISS)
                if label is not _MISS:
                    break

            if label is _MISS:
                if auto_label_stage1_missing and stage == "stage1":
                    label = stage1_default_class
                    auto_filled += 1
                else:
                    missing_hard.append((fp_norm, stage))
                    continue

            # (Optional) verify file exists
            if verify_files_exist:
                img_path = self.processed_root / fp_norm
                if not img_path.exists():
                    missing_files.append(str(img_path))
                    continue

            self.filepaths.append(fp_norm)
            self.labels.append(int(label))

        if auto_filled > 0:
            print(f"Auto-filled {auto_filled} missing Stage1 labels as class={stage1_default_class}")

        if missing_files:
            print(f"\nMissing image files for {len(missing_files)} samples (showing first 20):")
            for p in missing_files[:20]:
                print("  ", p)
            raise FileNotFoundError(f"Missing files for {len(missing_files)} samples.")

        if missing_hard:
            print(f"\nMissing labels for {len(missing_hard)} samples (showing first 30):")
            for fp_norm, stage in missing_hard[:30]:
                print("  ", fp_norm, "| stage:", stage)
            raise KeyError(f"Missing labels for {len(missing_hard)} samples.")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        rel_path = self.filepaths[idx]
        img_path = self.processed_root / rel_path

        img = Image.open(img_path)
        img = img.convert("L" if IMAGE_MODE == "gray" else "RGB")

        img = crop_to_tray_interior(img)

        if self.transform:
            img = self.transform(img)

        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, y

# =========================
# TRANSFORMS
# =========================
in_channels = 1 if IMAGE_MODE == "gray" else 3

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

if IMAGE_MODE == "gray":
    train_transform = T.Compose([
        T.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.95, 1.05)),
        T.RandomAffine(degrees=4, translate=(0.02, 0.02), scale=(0.97, 1.03)),
        T.ToTensor(),
        RandomBorderZero(p=0.7, min_frac=0.03, max_frac=0.10),
    ])
    val_transform = T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
    ])
else:
    train_transform = T.Compose([
        T.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.95, 1.05)),
        T.RandomAffine(degrees=4, translate=(0.02, 0.02), scale=(0.97, 1.03)),
        T.ColorJitter(brightness=0.08, contrast=0.12, saturation=0.03, hue=0.01),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        RandomBorderZero(p=0.7, min_frac=0.03, max_frac=0.10),
    ])
    val_transform = T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

# pick which fold you want to train on:
TRAIN_FOLD = "train3"  # "train1" / "train2" / "train3"

train_ds = ProcessedSplitDataset(
    index_rows, PROCESSED_ROOT, TRAIN_FOLD, train_label_map,
    transform=train_transform,
    auto_label_stage1_missing=False,   # IMPORTANT: prevent silent bias
    verify_files_exist=True
)

val_ds = ProcessedSplitDataset(
    index_rows, PROCESSED_ROOT, "val", val_label_map,
    transform=val_transform,
    auto_label_stage1_missing=False,
    verify_files_exist=True
)

print("Train dist:", Counter(train_ds.labels))
print("Val dist  :", Counter(val_ds.labels))

# -------------------------
# Class weights for loss
# -------------------------
counts = np.bincount(train_ds.labels, minlength=NUM_CLASSES)
counts = np.maximum(counts, 1)
class_weights = torch.tensor([1.0 / counts[i] for i in range(NUM_CLASSES)], dtype=torch.float).to(device)
print("Class counts:", counts)
print("Class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)

# =========================
# LOADERS
# =========================
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

x, y = next(iter(train_loader))
print("Batch image shape:", x.shape, "| Batch label shape:", y.shape)

# =========================
# MODEL
# =========================
class SimpleCNN_GAP(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = SimpleCNN_GAP(in_channels=in_channels, num_classes=NUM_CLASSES).to(device)

# (Optional) load your previous checkpoint

model.load_state_dict(
    torch.load("../../models/classifier/Stage1/model.pt", map_location=device),
    strict=False,
)


def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("NUM OF PARAMETERS:", count_parameters(model))

# Freeze feature extractor first
for p in model.features.parameters():
    p.requires_grad = False

# =========================
# OPTIMIZER
# =========================
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)

# =========================
# TRAIN LOOP
# =========================
CKPT_DIR = Path("../../models/classifier/Stage1/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    if epoch == UNFREEZE_EPOCH:
        print(f"\nUnfreezing features at epoch {epoch} ...")
        for p in model.features.parameters():
            p.requires_grad = True
        optimizer = optim.Adam(
            model.parameters(),
            lr=LR_FULL,
            weight_decay=WEIGHT_DECAY,
        )

    # ---- TRAIN ----
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / max(train_total, 1)
    train_acc  = train_correct / max(train_total, 1)

    # ---- VAL ----
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    preds_all, y_all, conf_all = [], [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(imgs)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            conf  = probs.max(dim=1).values

            val_loss_sum += loss.item() * imgs.size(0)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            preds_all.extend(preds.cpu().tolist())
            y_all.extend(labels.cpu().tolist())
            conf_all.extend(conf.cpu().tolist())

    val_loss = val_loss_sum / max(val_total, 1)
    val_acc  = val_correct / max(val_total, 1)

    ckpt_path = CKPT_DIR / f"{TRAIN_FOLD}_epoch_{epoch:03d}.pt"
    torch.save(
        {
            "epoch": epoch,
            "train_fold": TRAIN_FOLD,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "seed": SEED,
        },
        ckpt_path,
    )
    print(f"Saved checkpoint: {ckpt_path}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_path = CKPT_DIR / f"{TRAIN_FOLD}_best.pt"
        torch.save(model.state_dict(), best_path)
        print(f"New best model saved: {best_path} (val_loss={best_val_loss:.4f})")

    print(
        f"Epoch [{epoch}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f} "
        f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}"
    )
    print("Val preds dist:", Counter(preds_all))
    print("Val true  dist:", Counter(y_all))
    print("Val avg confidence:", float(np.mean(conf_all)))

    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for p, t in zip(preds_all, y_all):
        cm[t, p] += 1
    print("Confusion matrix (rows=true, cols=pred):\n", cm)

    if len(set(preds_all)) == 1:
        print("COLLAPSE DETECTED: model predicts only one class on validation")

torch.save(model.state_dict(), MODEL_OUT_PATH)
print("Saved model to:", MODEL_OUT_PATH)

In [ ]:
from collections import defaultdict

def basename_collisions(label_path: Path):
    data = json.load(open(label_path, "r"))
    full = []
    if isinstance(data, dict):
        for k,v in data.items():
            k = k.replace("\\","/")
            full.append((k.split("/")[-1], k, int(v)))
    else:
        for item in data:
            k = item["image"].replace("\\","/")
            # adapt if your list format differs
            empty = int(item["empty"])
            v = 0 if empty==1 else 1
            full.append((k.split("/")[-1], k, v))

    by = defaultdict(list)
    for bn, fp, v in full:
        by[bn].append((fp, v))

    collisions = {bn: lst for bn,lst in by.items() if len(lst) > 1}
    print("Basename collisions:", len(collisions))
    if collisions:
        bn = next(iter(collisions))
        print("Example:", bn)
        for fp,v in collisions[bn][:10]:
            print(" ", v, fp)

basename_collisions(TRAIN_LABELS_JSON)
basename_collisions(VAL_LABELS_JSON)

In [ ]:
# ============================================================
# Grad-CAM EXPORT (ALL CONV LAYERS) — batch-by-batch, no display
# - For each batch: compute preds once, then for EACH conv layer:
#   compute Grad-CAM per sample and save overlay to disk.
# - Saves: overlays/<layerXX>/<safe_rel_path>__T_P_conf.png
# - Writes a single CSV: gradcam_all_layers_summary.csv
# ============================================================

import os
import csv
import re
from pathlib import Path
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import cv2

import sys
sys.path.append("../..")
from src.xraygen.explain.gradcam import GradCAM

# -------------------------
# Output config
# -------------------------
OUT_DIR = Path("./gradcam_export_val_all_layers")
OVERLAY_DIR = OUT_DIR / "overlays"
OUT_CSV = OUT_DIR / "gradcam_all_layers_summary.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# FULL image transform for CAM (no crop)
# MUST match your model's expected input normalization
# -------------------------
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

cam_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),   # full image resize
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# -------------------------
# Helpers
# -------------------------
def overlay_heatmap(base_img_pil: Image.Image, heatmap01: np.ndarray, alpha=0.5):
    base_rgb = np.array(base_img_pil.convert("RGB"))
    heatmap = cv2.resize(heatmap01, (base_rgb.shape[1], base_rgb.shape[0]))
    heatmap = np.clip(heatmap, 0, 1)
    heatmap_u8 = np.uint8(255 * heatmap)

    heatmap_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    base_bgr = cv2.cvtColor(base_rgb, cv2.COLOR_RGB2BGR)
    overlay_bgr = cv2.addWeighted(base_bgr, 1 - alpha, heatmap_bgr, alpha, 0)
    overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
    return overlay_rgb

def safe_name(s: str) -> str:
    s = s.replace("\\", "/")
    s = re.sub(r"[^a-zA-Z0-9._\-\/]+", "_", s)
    return s.replace("/", "__")

def cleanup_layer_hooks(layer):
    # brute cleanup (prevents hook collisions when you recreate GradCAM per layer)
    try:
        layer._forward_hooks.clear()
        layer._backward_hooks.clear()
        layer._backward_pre_hooks.clear()
    except Exception:
        pass

# -------------------------
# Get ALL conv layers (order)
# -------------------------
model = model.to(device).eval()
conv_layers = [l for l in model.features if isinstance(l, torch.nn.Conv2d)]
assert len(conv_layers) > 0, "No Conv2d found in model.features"
print("Total Conv2d layers:", len(conv_layers))

# Create subfolders per layer
for li in range(len(conv_layers)):
    (OVERLAY_DIR / f"layer_{li:02d}").mkdir(parents=True, exist_ok=True)

# -------------------------
# CSV setup
# -------------------------
fieldnames = [
    "split",
    "rel_path",
    "abs_path",
    "true",
    "pred",
    "conf",
    "layer_idx",
    "layer_name",
    "saved_overlay",
    "status",
]

rows_out = []

# -------------------------
# Export loop (batch-by-batch)
# -------------------------
n_total = 0
n_saved = 0

# We do prediction under no_grad(), but GradCAM needs gradients.
# So: get preds/conf under no_grad, then enable_grad for CAM part.
for batch_idx, (imgs, y, paths) in enumerate(val_loader):
    print(f"\nBatch {batch_idx+1}/{len(val_loader)} | bs={len(paths)}")

    # ---- 1) Predict once for the batch
    imgs = imgs.to(device)
    y = y.to(device)

    with torch.no_grad():
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1)
        conf = probs.max(dim=1).values

    # Move small metadata to CPU lists
    true_list = [int(v) for v in y.detach().cpu().tolist()]
    pred_list = [int(v) for v in pred.detach().cpu().tolist()]
    conf_list = [float(v) for v in conf.detach().cpu().tolist()]
    paths_list = list(paths)

    # ---- 2) For EACH conv layer, compute CAM per sample (low memory)
    for li, layer in enumerate(conv_layers):
        layer_name = layer.__class__.__name__
        print(f"  Layer {li}/{len(conv_layers)-1}: {layer_name}")

        # Ensure no old hooks interfere
        cleanup_layer_hooks(layer)

        # Create GradCAM for this layer
        gradcam = GradCAM(model, layer)

        # Enable gradients for CAM
        with torch.enable_grad():
            for i in range(len(paths_list)):
                rel_path = paths_list[i]
                true_y = true_list[i]
                pred_y = pred_list[i]
                conf_i = conf_list[i]
                img_path = PROCESSED_ROOT / rel_path

                n_total += 1

                if not img_path.exists():
                    rows_out.append({
                        "split": "val",
                        "rel_path": rel_path,
                        "abs_path": str(img_path),
                        "true": true_y,
                        "pred": pred_y,
                        "conf": conf_i,
                        "layer_idx": li,
                        "layer_name": layer_name,
                        "saved_overlay": "",
                        "status": "missing_file",
                    })
                    continue

                # Load FULL image (no crop)
                try:
                    img_full = Image.open(img_path).convert("RGB")
                except Exception as e:
                    rows_out.append({
                        "split": "val",
                        "rel_path": rel_path,
                        "abs_path": str(img_path),
                        "true": true_y,
                        "pred": pred_y,
                        "conf": conf_i,
                        "layer_idx": li,
                        "layer_name": layer_name,
                        "saved_overlay": "",
                        "status": f"load_error:{type(e).__name__}",
                    })
                    continue

                # Prepare tensor (full image)
                x = cam_transform(img_full).unsqueeze(0).to(device)

                # IMPORTANT: GradCAM does backward internally; clear grads to avoid accumulation
                model.zero_grad(set_to_none=True)

                # Compute heatmap for predicted class
                try:
                    heatmap = gradcam(x, target_class=pred_y)  # expects np (H,W) in [0,1]
                    overlay = overlay_heatmap(img_full, heatmap, alpha=0.5)
                except Exception as e:
                    rows_out.append({
                        "split": "val",
                        "rel_path": rel_path,
                        "abs_path": str(img_path),
                        "true": true_y,
                        "pred": pred_y,
                        "conf": conf_i,
                        "layer_idx": li,
                        "layer_name": layer_name,
                        "saved_overlay": "",
                        "status": f"gradcam_error:{type(e).__name__}",
                    })
                    continue

                # Save overlay
                base = safe_name(rel_path)
                out_name = f"{base}__T{true_y}_P{pred_y}_conf{conf_i:.3f}.png"
                out_path = OVERLAY_DIR / f"layer_{li:02d}" / out_name

                overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
                ok = cv2.imwrite(str(out_path), overlay_bgr)

                if ok:
                    n_saved += 1
                    rows_out.append({
                        "split": "val",
                        "rel_path": rel_path,
                        "abs_path": str(img_path),
                        "true": true_y,
                        "pred": pred_y,
                        "conf": conf_i,
                        "layer_idx": li,
                        "layer_name": layer_name,
                        "saved_overlay": str(out_path),
                        "status": "ok",
                    })
                else:
                    rows_out.append({
                        "split": "val",
                        "rel_path": rel_path,
                        "abs_path": str(img_path),
                        "true": true_y,
                        "pred": pred_y,
                        "conf": conf_i,
                        "layer_idx": li,
                        "layer_name": layer_name,
                        "saved_overlay": "",
                        "status": "save_failed",
                    })

        # remove hooks for this GradCAM to avoid collisions
        if hasattr(gradcam, "remove"):
            gradcam.remove()
        else:
            cleanup_layer_hooks(layer)

    # Optional: flush intermediate CSV every batch (so you don't lose progress if it crashes)
    # This is safer for long runs.
    with open(OUT_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows_out)

    print(f"Batch done. Running totals: total_rows={n_total}, saved={n_saved}, csv={OUT_CSV}")

print("\nDONE")
print(f"Total layer-rows processed: {n_total}")
print(f"Total overlays saved: {n_saved}")
print(f"Outputs in: {OUT_DIR}")
print(f"CSV: {OUT_CSV}")
